# 03 · The training loop: modules, optimisers, diagnostics and checkpoints

*Base Model Training*, notebook 3 of 4

A training loop is a short sequence of explicit state changes: switch the model into training mode, clear the gradients, run the forward pass and the loss, backpropagate, update. Each line can go wrong in a way that still produces numbers. In this notebook you package your engine into reusable modules, write the Adam optimiser, write the update step itself, learn the checks that catch a broken loop, and save checkpoints that let an interrupted run continue *exactly* as if nothing had happened.

## What you will build

- `Linear`: a layer that owns its weights and lists them with `parameters()`;
- `dropout(x, p, training, rng)`: a layer that behaves differently in training and in evaluation;
- `adam_update`: Adam's update rule, with bias correction;
- `train_step`: one correct optimiser step, with a check for a non-finite loss;
- `grad_norm`: the size of all the gradients together;
- `optimiser_updates`: how many updates a training budget buys;
- `save_checkpoint` and `load_checkpoint`: everything needed to resume training exactly;
- 🧪 a tiny-batch diagnostic with deliberate bugs, a learning rate that explodes, and a resumption that forgets something.

## Learning objectives

By the end of this notebook you will be able to:

1. build modules that own their parameters, and switch them between training and evaluation modes;
2. implement SGD and Adam, and explain bias correction;
3. write a correct optimiser step and evaluate without updating the weights;
4. diagnose a training loop with a tiny-batch test, finite-loss checks, gradient norms and weight changes;
5. estimate the number of updates and the memory a run needs;
6. save and restore weights, optimiser state, training position and random state, and prove that resumption is exact.

### How to work through this notebook

Run the cells from top to bottom. At each **✍️ Your turn** you type a function yourself, then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook carries on with a hidden reference version, so *Run All* always works and you can come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for one plot of the final run; the numbers behind the plot are also printed. The standard-library modules `json`, `os`, `tempfile` and `time` store random-generator state, build file paths, provide a scratch folder and time a short run. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Checkpoints are written to a temporary folder that the operating system cleans up, so the notebook leaves nothing behind in your workspace.

## Setup

Run this cell first. It imports the libraries, creates a seeded random generator and defines `practice`, the helper that checks each function you write against its reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import json
import os
import tempfile
import time

import matplotlib.pyplot as plt

As in the earlier notebooks, `practice_via` checks a function through a **probe**, a small function that runs your code on a fresh example and returns plain numbers to compare. In this notebook the probes build fresh models, optimisers and random generators from seeds, so that your code and the reference start from identical states.

In [ ]:
def practice_via(name, yours, reference, probe, cases, atol=1e-6):
    """Like practice, but compares probe(function, *case) for your function and for the reference.

    Use it when a function works by changing objects (filling in .grad, updating
    weights) or when several answers are correct: the probe runs the function on
    a fresh example and turns the outcome into plain numbers. Returns your
    function once the measurements match, otherwise the reference.
    """
    def measure_yours(*case):
        return probe(yours, *case)

    def measure_reference(*case):
        return probe(reference, *case)

    chosen = practice(name, measure_yours, measure_reference, cases, atol)
    return yours if chosen is measure_yours else reference

## 📦 Recap: what you built earlier

The hidden cell below defines the tiny tensor engine from notebook 1 (`Tensor`, `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`), followed by the data pieces from notebook 2: `make_readings`, `group_split`, `fit_standardiser` and `standardise`. Run it; open it only if you want to look back at the code. The next cell rebuilds notebook 2's data and split.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

# ── From notebook 2: generated data, split by source, standardiser ─────────────
def make_readings(n_sources, rows_per_source, rng, spread=0.3, label_noise=0.1, flipped_fraction=0.0):
    """Generated sensor readings. Returns X (rows, 2) raw measurements, y (rows,) labels in {0, 1}
    and groups (rows,) source ids; the rows of one source cluster around that source's centre."""
    centres = rng.normal(size=(n_sources, 2))
    flipped = rng.random(n_sources) < flipped_fraction
    groups = np.repeat(np.arange(n_sources), rows_per_source)
    signal = centres[groups] + spread * rng.normal(size=(groups.size, 2))
    noisy = rng.random(groups.size) < label_noise
    y = ((signal[:, 0] * signal[:, 1] > 0) ^ flipped[groups] ^ noisy).astype(int)
    X = np.column_stack([50.0 + 10.0 * signal[:, 0], 0.5 + 0.1 * signal[:, 1]])
    return X, y, groups


def group_split(groups, fractions, rng):
    sources = np.unique(groups)
    shuffled = rng.permutation(sources)
    n_train = round(fractions[0] * len(sources))
    n_val = round(fractions[1] * len(sources))
    parts = shuffled[:n_train], shuffled[n_train:n_train + n_val], shuffled[n_train + n_val:]
    return tuple(np.flatnonzero(np.isin(groups, part)) for part in parts)


def fit_standardiser(X):
    return X.mean(axis=0), np.maximum(X.std(axis=0), 1e-6)


def standardise(X, mean, scale):
    return (X - mean) / scale

In [ ]:
X, y, groups = make_readings(160, 5, np.random.default_rng(0))
train_rows, val_rows, test_rows = group_split(groups, (0.6, 0.2, 0.2), np.random.default_rng(1))
mean, scale = fit_standardiser(X[train_rows])
Z = standardise(X, mean, scale)
print(f"{train_rows.size} training, {val_rows.size} validation and {test_rows.size} test rows; inputs standardised")

## 1 · Modules: layers that own their parameters

In notebook 2 the weights were a bare list, `[W1, b1, W2, b2]`, and the forward pass had to know their order. A **module** bundles parameters with the computation that uses them. Two methods carry the whole idea:

- calling the module, `layer(x)`, runs its forward pass;
- `layer.parameters()` returns the list of tensors that an optimiser should update.

Larger modules are built from smaller ones, and their `parameters()` simply concatenates the lists of their parts. Frameworks such as PyTorch work the same way: a `torch.nn.Linear` layer holds a weight and a bias, and `model.parameters()` collects them from every layer.

In [ ]:
init_demo = np.random.default_rng(2)
W_demo, b_demo = parameter(2, 3, rng=init_demo), zeros_parameter(3)
x_demo = Tensor(Z[train_rows[:4]])
print("x @ W + b has shape", (x_demo @ W_demo + b_demo).shape, "| W", W_demo.shape, "| b", b_demo.shape)

### ✍️ Your turn: `Linear`

Write the class `Linear`:

- `__init__(self, n_in, n_out, rng)` creates `self.W = parameter(n_in, n_out, rng=rng)` (He initialisation) and `self.b = zeros_parameter(n_out)`;
- `__call__(self, x)` returns `x @ self.W + self.b` for a `Tensor` `x` of shape `(batch, n_in)`;
- `parameters(self)` returns `[self.W, self.b]`, weights first.

The probe builds a layer from a seed, runs it on a batch, backpropagates a fixed upstream gradient, and compares the outputs, the gradients and the parameter shapes with the reference.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
class Linear_reference:
    def __init__(self, n_in, n_out, rng):
        self.W = parameter(n_in, n_out, rng=rng)
        self.b = zeros_parameter(n_out)

    def __call__(self, x):
        return x @ self.W + self.b

    def parameters(self):
        return [self.W, self.b]

In [ ]:
class Linear:
    """A dense layer: x @ W + b, with W of shape (n_in, n_out) and b of shape (n_out,)."""

    def __init__(self, n_in, n_out, rng):
        # ✍️ Your code here (two lines)
        raise NotImplementedError

    def __call__(self, x):
        # ✍️ Your code here (one line)
        raise NotImplementedError

    def parameters(self):
        # ✍️ Your code here (one line)
        raise NotImplementedError

In [ ]:
def layer_probe(layer_cls, n_in, n_out, seed, x):
    """Probe: build a layer from a seed, run it on x, backpropagate a fixed upstream gradient,
    and return the output, the gradients and the parameter shapes as plain values."""
    layer = layer_cls(n_in, n_out, np.random.default_rng(seed))
    inputs = Tensor(x, requires_grad=True)
    out = layer(inputs)
    out.backward(np.cos(np.arange(out.data.size) + 1.0).reshape(out.shape))
    params = layer.parameters()
    return (out.data, inputs.grad, *[p.grad for p in params], [p.shape for p in params])


Linear = practice_via("Linear", Linear, Linear_reference, layer_probe, cases=[
    (2, 3, 0, Z[train_rows[:4]]),
    (3, 1, 1, np.random.default_rng(3).normal(size=(5, 3))),
    (2, 4, 2, Z[train_rows[:1]]),                                  # edge case: a batch of one
])

**Expected behaviour.** ✅ once your layer creates its weights in the same order as the reference (weights first, then the bias, which draws no random numbers), returns `x @ W + b`, and lists `[W, b]`. The gradient of `b` is the upstream gradient summed over the batch, which your engine's `unbroadcast` provides.

## 2 · Dropout, and the difference between training and evaluation

**Dropout** sets each hidden activation to zero with probability $p$ during training, so the network cannot rely on any single unit. To keep the expected value of each activation unchanged, the surviving entries are divided by $1 - p$ ("inverted" dropout):

$$\operatorname{dropout}(x)_i = \frac{m_i\, x_i}{1 - p}, \qquad m_i = 1 \text{ with probability } 1 - p, \text{ otherwise } 0.$$

During evaluation dropout must do nothing, because validation scores must not depend on random masks. A module therefore carries a **mode**: `model.train()` before each update and `model.eval()` before validation or prediction. In frameworks such as PyTorch the mode also changes batch normalisation. The mode does *not* control gradient recording; PyTorch uses a separate `torch.no_grad()` context for that. Our engine records every operation on a tensor that requires a gradient, but evaluation never calls `backward()`, so nothing is updated and the recorded graph is simply discarded.

In [ ]:
mask_rng = np.random.default_rng(4)
keep = mask_rng.random(10) >= 0.5        # True means "keep this entry"
print("mask:", keep.astype(int), "| kept entries are scaled by", 1 / (1 - 0.5))
print("mean of 100,000 masked-and-scaled ones:", (((mask_rng.random(100_000) >= 0.5) / 0.5)).mean().round(3))

### ✍️ Your turn: `dropout`

Write `dropout(x, p, training, rng)`:

- **inputs:** `x`, a `Tensor`; `p`, the probability of dropping an entry; `training`, a bool; `rng`, a NumPy generator (it may be `None` in evaluation);
- **output:** `x` itself when `training` is `False` or `p == 0`; otherwise `x * (keep / (1 - p))`, where `keep = rng.random(x.shape) >= p` is drawn with exactly one call.

Hint: multiplying a `Tensor` by a NumPy array lifts the array into a constant, so the mask itself receives no gradient.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def dropout_reference(x, p, training, rng):
    if not training or p == 0:
        return x
    keep = rng.random(x.shape) >= p
    return x * (keep / (1.0 - p))

In [ ]:
def dropout(x, p, training, rng):
    """Inverted dropout: in training, zero each entry with probability p and divide the rest by 1 - p."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
def dropout_probe(dropout_fn, x, p, training, seed):
    """Probe: apply dropout to a fresh tensor with a fresh generator; return (output, input gradient)."""
    inputs = Tensor(x, requires_grad=True)
    out = dropout_fn(inputs, p, training, np.random.default_rng(seed))
    out.backward(np.ones(out.shape))
    return out.data, inputs.grad


drop_rng = np.random.default_rng(5)
dropout = practice_via("dropout", dropout, dropout_reference, dropout_probe, cases=[
    (drop_rng.normal(size=(4, 5)), 0.5, True, 0),
    (drop_rng.normal(size=(4, 5)), 0.5, False, 0),       # evaluation: nothing changes
    (drop_rng.normal(size=(2, 4)), 0.25, True, 1),
    (drop_rng.normal(size=(3, 3)), 0.0, True, 2),        # edge case: p = 0 in training
])

**Expected behaviour.** In training, about half the entries become 0 and the rest double when $p = 0.5$; the gradient follows the same mask, because a dropped unit contributed nothing. In evaluation the output equals the input exactly. The next cell builds a multilayer perceptron from your `Linear` and `dropout`; read it, because every later cell uses it.

In [ ]:
class MLP:
    """Dense layers with ReLU between them, and dropout after each hidden ReLU while training."""

    def __init__(self, sizes, rng, dropout_p=0.0):
        self.sizes, self.dropout_p = list(sizes), dropout_p
        self.layers = [Linear(n_in, n_out, rng) for n_in, n_out in zip(sizes[:-1], sizes[1:])]
        self.training = True

    def __call__(self, x, rng=None):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = dropout(x.relu(), self.dropout_p, self.training, rng)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def train(self):
        self.training = True

    def eval(self):
        self.training = False


model = MLP([2, 16, 2], np.random.default_rng(6), dropout_p=0.5)
batch = Tensor(Z[train_rows[:3]])
print("parameters:", [p.shape for p in model.parameters()], "->", sum(p.data.size for p in model.parameters()), "numbers")
model.train()
print("training mode, two calls:\n", model(batch, np.random.default_rng(7)).data.round(3), "\n", model(batch, np.random.default_rng(8)).data.round(3))
model.eval()
print("evaluation mode, two calls agree:", np.array_equal(model(batch).data, model(batch).data))

## 3 · Optimisers: SGD and Adam

An optimiser turns gradients into updates. **Stochastic gradient descent** (SGD) with momentum keeps a velocity $v$, a running sum of past gradients that smooths out noise:

$$v \leftarrow \mu v + g, \qquad \theta \leftarrow \theta - \eta v .$$

**Adam** keeps two running averages per parameter, of the gradient ($m$) and of its square ($v$), and divides one by the square root of the other, so that each parameter gets a step size of roughly $\eta$, whatever the scale of its gradient:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g, \qquad v_t = \beta_2 v_{t-1} + (1 - \beta_2) g^2,$$
$$\hat m_t = \frac{m_t}{1 - \beta_1^t}, \qquad \hat v_t = \frac{v_t}{1 - \beta_2^t}, \qquad \theta \leftarrow \theta - \eta\, \frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}.$$

The **bias correction** with $1 - \beta^t$ matters early on. Both averages start at zero, so after one step $m_1 = 0.1\,g$ with $\beta_1 = 0.9$: ten times too small. Dividing by $1 - 0.9^1 = 0.1$ restores $g$. As $t$ grows, $\beta^t \to 0$ and the correction fades away. The step counter $t$ is therefore part of the optimiser's state, alongside $m$ and $v$.

Both optimisers below update the weights **in place** (`p.data[...] = ...` or `p.data -= ...`), as framework optimisers do; notebook 4 shows why that matters when you keep a copy of the best weights. Parameters without a gradient are skipped.

In [ ]:
class SGD:
    """Plain or momentum gradient descent. Parameters without a gradient are left alone."""

    def __init__(self, params, lr, momentum=0.0):
        self.params, self.lr, self.momentum = list(params), lr, momentum
        self.velocity = [np.zeros_like(p.data) for p in self.params]

    def zero_grad(self):
        zero_grad(self.params)

    def step(self):
        for p, velocity in zip(self.params, self.velocity):
            if p.grad is None:
                continue
            velocity *= self.momentum
            velocity += p.grad
            p.data -= self.lr * velocity

    def state_dict(self):
        return {f"velocity{i}": velocity for i, velocity in enumerate(self.velocity)}

    def load_state_dict(self, state):
        for i, velocity in enumerate(self.velocity):
            velocity[...] = state[f"velocity{i}"]


g_demo = np.array([0.5, -2.0, 1e-3])
print("m after one step without correction:", 0.1 * g_demo, "| with correction:", 0.1 * g_demo / (1 - 0.9))

### ✍️ Your turn: `adam_update`

Write the pure function `adam_update(param, grad, m, v, t, lr, beta1=0.9, beta2=0.999, eps=1e-8)`:

- **inputs:** arrays `param`, `grad`, `m` and `v` of the same shape; `t`, the step number, starting at 1; `lr`, `beta1`, `beta2` and `eps` as in the formulas;
- **output:** the tuple `(new_param, new_m, new_v)`. Return new arrays and do not modify the inputs, so that the check can compare return values.

Hint: four lines compute $m_t$, $v_t$, $\hat m_t$ and $\hat v_t$; the fifth returns the tuple. `np.sqrt` takes the square root.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def adam_update_reference(param, grad, m, v, t, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    m = beta1 * m + (1 - beta1) * grad
    v = beta2 * v + (1 - beta2) * grad ** 2
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    return param - lr * m_hat / (np.sqrt(v_hat) + eps), m, v

In [ ]:
def adam_update(param, grad, m, v, t, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    """One Adam step with bias correction. Returns (new_param, new_m, new_v) without modifying the inputs."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
adam_rng = np.random.default_rng(9)
adam_update = practice("adam_update", adam_update, adam_update_reference, cases=[
    (np.ones(3), np.array([0.5, -2.0, 1e-3]), np.zeros(3), np.zeros(3), 1, 0.1),       # the first step
    (adam_rng.normal(size=(2, 3)), adam_rng.normal(size=(2, 3)), adam_rng.normal(size=(2, 3)) * 0.1,
     adam_rng.uniform(0.0, 0.1, (2, 3)), 10, 0.01),
    (np.ones(2), np.zeros(2), np.zeros(2), np.zeros(2), 1, 0.1),                       # edge case: zero gradient
])

**Expected behaviour.** On the first step every parameter moves by almost exactly the learning rate, 0.1, in the direction opposite to its gradient, whether the gradient is 0.5, $-2$ or 0.001: Adam's step size does not depend on the gradient's scale. With a zero gradient and zero moments nothing moves. The class below wraps your function and stores $m$, $v$ and $t$.

In [ ]:
class Adam:
    """Adam: per-parameter step sizes from running averages of the gradient and its square."""

    def __init__(self, params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.beta1, self.beta2, self.eps = list(params), lr, beta1, beta2, eps
        self.m = [np.zeros_like(p.data) for p in self.params]
        self.v = [np.zeros_like(p.data) for p in self.params]
        self.t = 0

    def zero_grad(self):
        zero_grad(self.params)

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            new_value, self.m[i], self.v[i] = adam_update(p.data, p.grad, self.m[i], self.v[i], self.t,
                                                          self.lr, self.beta1, self.beta2, self.eps)
            p.data[...] = new_value      # in place, as framework optimisers do

    def state_dict(self):
        state = {"t": np.array(self.t)}
        for i, (m, v) in enumerate(zip(self.m, self.v)):
            state[f"m{i}"], state[f"v{i}"] = m, v
        return state

    def load_state_dict(self, state):
        self.t = int(state["t"])
        for i in range(len(self.params)):
            self.m[i] = np.array(state[f"m{i}"], dtype=np.float64)
            self.v[i] = np.array(state[f"v{i}"], dtype=np.float64)

## 4 · One correct optimiser step

Every update in this course follows the same five moves, in this order:

1. `model.train()`, so that dropout is active;
2. `optimiser.zero_grad()`, because `backward()` *adds* to existing gradients;
3. the forward pass and the loss;
4. `loss.backward()`, which fills in the gradients and changes no weights;
5. `optimiser.step()`, which changes the weights using those gradients and its own state.

Between steps 3 and 4 a cheap guard pays for itself: if the loss is `nan` or infinite, stop at once with a clear message, before the bad numbers spread into every weight. Validation follows a different recipe: `model.eval()`, a forward pass, and no `backward()` or `step()` at all.

Gradient accumulation deliberately skips step 2 for a few micro-batches and calls `step()` once, but that is a design decision with its own normalisation (Exercise 3), not something to happen by accident.

### ✍️ Your turn: `train_step`

Write `train_step(model, optimiser, X, y, rng)`:

- **inputs:** `model`, an `MLP`; `optimiser`, an `SGD` or `Adam`; `X`, a NumPy array of inputs, shape `(batch, features)`; `y`, integer targets; `rng`, the generator for dropout masks;
- **effect and output:** the five moves above, with `loss = cross_entropy(model(Tensor(X), rng), y)`; if `np.isfinite(loss.data)` is `False`, raise `FloatingPointError` with a helpful message instead of calling `backward()`; otherwise return the loss as a Python `float`.

The probe starts a fresh model in *evaluation* mode, so your step must switch it to training mode itself. It runs three steps and compares the losses and the final weights. A third case feeds `nan` inputs and expects your function to raise `FloatingPointError`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def train_step_reference(model, optimiser, X, y, rng):
    model.train()
    optimiser.zero_grad()
    loss = cross_entropy(model(Tensor(X), rng), y)
    if not np.isfinite(loss.data):
        raise FloatingPointError(f"non-finite training loss ({float(loss.data)}): check the learning rate and the data")
    loss.backward()
    optimiser.step()
    return float(loss.data)

In [ ]:
def train_step(model, optimiser, X, y, rng):
    """One optimiser step on the batch (X, y); returns the loss as a float, or raises FloatingPointError."""
    # ✍️ Your code here (about eight lines)
    raise NotImplementedError

In [ ]:
def step_probe(step_fn, seed, inputs, targets):
    """Probe: three steps on a fresh model that starts in evaluation mode; returns the losses and the final weights."""
    model = MLP([2, 8, 2], np.random.default_rng(seed), dropout_p=0.2)
    model.eval()                                  # train_step must switch to training mode itself
    optimiser = Adam(model.parameters(), lr=0.01)
    step_rng = np.random.default_rng(seed + 100)
    try:
        losses = [step_fn(model, optimiser, inputs, targets, step_rng) for _ in range(3)]
    except FloatingPointError:
        return "stopped with FloatingPointError"
    return (losses, *[p.data for p in model.parameters()])


train_step = practice_via("train_step", train_step, train_step_reference, step_probe, cases=[
    (0, Z[train_rows[:16]], y[train_rows[:16]]),
    (1, Z[train_rows[16:48]], y[train_rows[16:48]]),
    (2, np.full((4, 2), np.nan), np.array([0, 1, 0, 1])),        # edge case: a broken batch must stop training
])


def evaluate(model, rows):
    """Validation recipe: evaluation mode, a forward pass, no backward and no step. Returns (loss, accuracy)."""
    model.eval()
    logits = model(Tensor(Z[rows]))
    return float(cross_entropy(logits, y[rows]).data), float((logits.data.argmax(axis=1) == y[rows]).mean())

**Expected behaviour.** ✅ once your step follows the five moves in order. If you forget `model.train()`, dropout stays off and the losses differ; if you forget `zero_grad()`, the second and third steps use accumulated gradients and the weights differ; and without the guard, the `nan` case returns `nan` losses instead of stopping. `evaluate`, defined in the same cell, is the validation recipe.

## 5 · Diagnose before you train for long

Long runs hide bugs, so check the wiring on something small first.

**Overfit a tiny batch.** A network with a few thousand parameters should be able to memorise 16 correctly labelled examples, driving their loss close to zero. If it cannot, something in the loop is broken: detached gradients, a missing `zero_grad`, a learning rate far too small or far too large, or targets that do not match the outputs. Success proves only that the loop is wired correctly; it says nothing about generalisation.

**Watch the numbers during training.** Check that the loss is finite, that the **gradient norm** $\lVert g \rVert = \sqrt{\sum_p \sum_i g_{p,i}^2}$ over all parameters is finite and not exploding, that the weights actually change after an update, and that the predictions do not collapse to a single class.

In [ ]:
tiny = train_rows[:16]
tiny_model = MLP([2, 32, 32, 2], np.random.default_rng(3))
tiny_optimiser = Adam(tiny_model.parameters(), lr=0.01)
before = [p.data.copy() for p in tiny_model.parameters()]
tiny_losses = [train_step(tiny_model, tiny_optimiser, Z[tiny], y[tiny], None) for _ in range(300)]
tiny_model.eval()
tiny_accuracy = (tiny_model(Tensor(Z[tiny])).data.argmax(axis=1) == y[tiny]).mean()
moved = sum(float(np.abs(p.data - b).sum()) for p, b in zip(tiny_model.parameters(), before))
print(f"tiny batch: loss {tiny_losses[0]:.3f} -> {tiny_losses[-1]:.5f}, accuracy {tiny_accuracy:.0%}, weights moved by {moved:.1f}")
assert tiny_losses[-1] < 0.05 and tiny_accuracy == 1.0, "the loop cannot memorise 16 examples: look for a wiring bug"

### ✍️ Your turn: `grad_norm`

Write `grad_norm(params)`:

- **input:** a list of tensors, some of which may have `grad = None`;
- **output:** a Python `float`, the square root of the sum of the squares of every gradient entry, skipping tensors without a gradient (an empty sum gives 0.0).

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def grad_norm_reference(params):
    return float(np.sqrt(sum(float((p.grad ** 2).sum()) for p in params if p.grad is not None)))

In [ ]:
def grad_norm(params):
    """Global L2 norm of all gradients, skipping tensors whose grad is None."""
    # ✍️ Your code here (one or two lines)
    raise NotImplementedError

In [ ]:
def tensor_with_grad(grad):
    tensor = Tensor(np.zeros(np.shape(grad)), requires_grad=True)
    tensor.grad = None if grad is None else np.array(grad, dtype=float)
    return tensor


grad_norm = practice("grad_norm", grad_norm, grad_norm_reference, cases=[
    ([tensor_with_grad([[3.0, 0.0], [0.0, 4.0]])],),                          # a norm of 5
    ([tensor_with_grad([1.0, 2.0]), tensor_with_grad([[2.0]]), tensor_with_grad(None)],),
    ([tensor_with_grad(None)],),                                               # edge case: no gradients at all
])
print(f"gradient norm after the tiny-batch run: {grad_norm(tiny_model.parameters()):.2e}")

**Expected behaviour.** The first case gives 5, the second 3 (the tensor without a gradient is skipped), and the third 0. After the tiny batch has been memorised the gradients are tiny, which is what a converged run looks like. In a healthy run the norm fluctuates within a range; a sudden jump by orders of magnitude announces an explosion before the loss turns `nan`.

## 6 · Plan the budget: updates, time and memory

An **epoch** is one pass over the training rows; an **update** (optimiser step) changes the weights once. With **gradient accumulation**, several micro-batches contribute to one update, so the *effective* batch size is the micro-batch size times the number of accumulated micro-batches. Report updates and examples processed, not just epochs: two runs with "3 epochs" can differ by a factor of ten in the number of updates when their batch sizes differ.

What happens to an incomplete final batch is a policy. With `drop_last=True`, a final partial micro-batch, and a final incomplete group of micro-batches, are skipped; with `drop_last=False` they still count, as smaller updates.

### ✍️ Your turn: `optimiser_updates`

Write `optimiser_updates(n_examples, micro_batch, accumulation, epochs, drop_last=True)`:

- **output:** the total number of optimiser updates, an `int`.
- With `drop_last=True`: micro-batches per epoch `= n_examples // micro_batch`, and updates per epoch `= micro_batches // accumulation`.
- With `drop_last=False`: both divisions round **up** instead (ceiling division, `-(-a // b)`).
- Multiply the updates per epoch by `epochs`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def optimiser_updates_reference(n_examples, micro_batch, accumulation, epochs, drop_last=True):
    if drop_last:
        micro_batches = n_examples // micro_batch
        per_epoch = micro_batches // accumulation
    else:
        micro_batches = -(-n_examples // micro_batch)
        per_epoch = -(-micro_batches // accumulation)
    return epochs * per_epoch

In [ ]:
def optimiser_updates(n_examples, micro_batch, accumulation, epochs, drop_last=True):
    """Number of optimiser updates in a run, under the given final-batch policy."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
optimiser_updates = practice("optimiser_updates", optimiser_updates, optimiser_updates_reference, cases=[
    (8192, 16, 4, 3),               # the lesson's example
    (1000, 16, 4, 1),
    (1000, 16, 4, 1, False),
    (10, 16, 1, 5),                 # edge case: fewer examples than one micro-batch
])

**Expected behaviour.** The lesson's example, 8,192 examples in micro-batches of 16 with 4 accumulated per update, gives 512 micro-batches and 128 updates per epoch: 384 updates in 3 epochs, each update seeing an effective batch of 64. For 1,000 examples the policy matters: 15 updates with `drop_last=True` and 16 without. With only 10 examples and `drop_last=True`, the run makes **no** updates at all, a silent failure that this arithmetic catches before you wait for nothing.

The next cell times 50 real steps and multiplies, which is how to estimate a run's duration before starting it. It then measures the memory that training state takes.

In [ ]:
timing_model = MLP([2, 32, 32, 2], np.random.default_rng(8), dropout_p=0.1)
timing_optimiser, timing_rng = Adam(timing_model.parameters()), np.random.default_rng(9)
start = time.perf_counter()
for _ in range(50):
    rows = train_rows[timing_rng.choice(train_rows.size, 32, replace=False)]
    train_step(timing_model, timing_optimiser, Z[rows], y[rows], timing_rng)
seconds_per_step = (time.perf_counter() - start) / 50
planned = optimiser_updates(train_rows.size, 32, 1, 40)
print(f"{1000 * seconds_per_step:.2f} ms per step; 40 epochs of batches of 32 = {planned} updates "
      f"= {planned * 32:,} examples, about {planned * seconds_per_step:.1f} s on this machine")

memory_model = MLP([2, 64, 64, 2], np.random.default_rng(10))
memory_optimiser = Adam(memory_model.parameters())
train_step(memory_model, memory_optimiser, Z[train_rows[:32]], y[train_rows[:32]], None)
n_params = sum(p.data.size for p in memory_model.parameters())
weight_bytes = sum(p.data.nbytes for p in memory_model.parameters())
grad_bytes = sum(p.grad.nbytes for p in memory_model.parameters())
moment_bytes = sum(m.nbytes + v.nbytes for m, v in zip(memory_optimiser.m, memory_optimiser.v))
total_bytes = weight_bytes + grad_bytes + moment_bytes
print(f"{n_params:,} parameters: weights {weight_bytes:,} B + gradients {grad_bytes:,} B + Adam moments {moment_bytes:,} B "
      f"= {total_bytes / n_params:.0f} bytes per parameter, before activations")
assert total_bytes == 32 * n_params
for name, count in [("125 million", 125e6), ("1 billion", 1e9), ("7 billion", 7e9)]:
    print(f"{name:>12} parameters: float64 Adam {32 * count / 1e9:6.0f} GB | mixed-precision Adam {16 * count / 1e9:5.0f} GB")

**What to notice.** Our engine stores everything in `float64`, so weights, gradients and Adam's two moments take $4 \times 8 = 32$ bytes per parameter. A common mixed-precision recipe stores 2-byte weights and gradients, plus 4-byte master weights and two 4-byte moments: 16 bytes per parameter. A 7-billion-parameter model would then need about 112 GB for its training state alone, before a single activation is stored for the backward pass. Real frameworks lay this out differently, so measure the configuration you actually run.

## 7 · Checkpoints that resume exactly

A file that holds only the weights is enough for *prediction* (together with the architecture and the preprocessing; notebook 4 covers that). To *resume training exactly*, a checkpoint needs everything the next update depends on:

- the **weights**;
- the **optimiser state**: Adam's $m$, $v$ and step counter $t$;
- the **training position**: the step number, which also drives the learning-rate schedule;
- the **random state**: the generator that draws batches and dropout masks.

This notebook draws every mini-batch independently from the generator, so the step number and the generator's state are the whole training position. An epoch-based loader, like `shuffled_batches` from notebook 2, would also have to save the epoch's order and how far through it the run had got.

`np.savez` writes named arrays into one `.npz` file. A generator's state is a small dictionary of integers; `json.dumps` turns it into a string, which NumPy stores as an ordinary text array. That matters: `np.load` refuses arrays of arbitrary Python objects by default (`allow_pickle=False`), because unpickling a file can run code. Keep that default, and load only checkpoint files from sources you trust.

In [ ]:
checkpoint_dir = tempfile.mkdtemp(prefix="base_models_")
state = np.random.default_rng(42).bit_generator.state
print("a generator's state:", state)
print("as JSON text:", json.dumps(state)[:70], "...")

### ✍️ Your turn: `save_checkpoint`

Write `save_checkpoint(path, model, optimiser, step, rng)`, which calls `np.savez(path, ...)` once with these names:

- `param_0`, `param_1`, … for `p.data` of each tensor in `model.parameters()`, in order;
- `opt_` followed by each key of `optimiser.state_dict()`, such as `opt_t`, `opt_m0` and `opt_v0`;
- `step`, the step number;
- `rng_state`, the text `json.dumps(rng.bit_generator.state)`.

Hint: build one dictionary of arrays and call `np.savez(path, **arrays)`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def save_checkpoint_reference(path, model, optimiser, step, rng):
    arrays = {f"param_{i}": p.data for i, p in enumerate(model.parameters())}
    arrays.update({f"opt_{key}": value for key, value in optimiser.state_dict().items()})
    np.savez(path, step=step, rng_state=json.dumps(rng.bit_generator.state), **arrays)

In [ ]:
def save_checkpoint(path, model, optimiser, step, rng):
    """Save weights, optimiser state, step and random state to the .npz file at path."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

### ✍️ Your turn: `load_checkpoint`

Write `load_checkpoint(path, model, optimiser, rng)`, the mirror image:

- copy each `param_i` into the matching parameter **in place**, with `p.data[...] = saved[f"param_{i}"]`, so that the optimiser's references to the parameters stay valid;
- pass `optimiser.load_state_dict` a dictionary of every saved key that starts with `opt_`, with that prefix removed;
- restore the generator with `rng.bit_generator.state = json.loads(str(saved["rng_state"]))`;
- return `int(saved["step"])`.

Hint: `with np.load(path) as saved:` opens the file and closes it afterwards; `saved.files` lists the names.

The probe trains a small model for three steps, saves it, loads it into a *fresh* model, optimiser and generator built from different seeds, and compares what came back. Your `save_checkpoint` is checked with the reference loader, and your `load_checkpoint` with the saver that passed.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def load_checkpoint_reference(path, model, optimiser, rng):
    with np.load(path) as saved:
        for i, p in enumerate(model.parameters()):
            p.data[...] = saved[f"param_{i}"]
        optimiser.load_state_dict({key[4:]: saved[key] for key in saved.files if key.startswith("opt_")})
        rng.bit_generator.state = json.loads(str(saved["rng_state"]))
        return int(saved["step"])

In [ ]:
def load_checkpoint(path, model, optimiser, rng):
    """Restore weights, optimiser state and random state saved by save_checkpoint; return the step."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
def round_trip(save_fn, load_fn, seed):
    """Probe: train briefly, save with save_fn, load into fresh objects with load_fn, and report what came back."""
    model = MLP([2, 8, 2], np.random.default_rng(seed), dropout_p=0.1)
    optimiser = Adam(model.parameters(), lr=0.01)
    step_rng = np.random.default_rng(seed + 1)
    for _ in range(3):
        train_step(model, optimiser, Z[train_rows[:32]], y[train_rows[:32]], step_rng)
    path = os.path.join(checkpoint_dir, f"probe_{seed}.npz")
    save_fn(path, model, optimiser, 3, step_rng)
    fresh_model = MLP([2, 8, 2], np.random.default_rng(seed + 50), dropout_p=0.1)
    fresh_optimiser = Adam(fresh_model.parameters(), lr=0.01)
    fresh_rng = np.random.default_rng(seed + 99)
    step = load_fn(path, fresh_model, fresh_optimiser, fresh_rng)
    return (step, fresh_optimiser.t, fresh_rng.random(3), *[p.data for p in fresh_model.parameters()],
            *fresh_optimiser.m, *fresh_optimiser.v)


save_checkpoint = practice_via("save_checkpoint", save_checkpoint, save_checkpoint_reference,
                               lambda fn, seed: round_trip(fn, load_checkpoint_reference, seed), cases=[(0,), (1,)])
load_checkpoint = practice_via("load_checkpoint", load_checkpoint, load_checkpoint_reference,
                               lambda fn, seed: round_trip(save_checkpoint, fn, seed), cases=[(0,), (1,)])

**Expected behaviour.** Two ✅. After loading, the fresh model holds the trained weights, the fresh optimiser reports `t = 3` with the saved moments, and the fresh generator produces exactly the random numbers that the original would have produced next.

### The proof: an interrupted run equals an uninterrupted one

Run A trains for $2N$ steps without stopping. Run B trains for $N$ steps and saves a checkpoint, and then everything is thrown away, as if the process had stopped. A fresh model, optimiser and generator, deliberately built from *different* seeds, load the checkpoint and train for $N$ more steps. If the checkpoint is complete, the final weights are not just close but **identical**, bit for bit. The run uses dropout, so the random state matters, and a learning-rate warm-up, so the step number matters too.

In [ ]:
def lr_at(step, base=0.01, warmup=20):
    """Linear warm-up over the first `warmup` steps, then constant."""
    return base * min(1.0, step / warmup)


def run_training(model, optimiser, rng, start_step, end_step, batch_size=32):
    """Run steps start_step + 1 to end_step; every step draws its batch and dropout masks from rng."""
    for step in range(start_step + 1, end_step + 1):
        optimiser.lr = lr_at(step)
        rows = train_rows[rng.choice(train_rows.size, batch_size, replace=False)]
        train_step(model, optimiser, Z[rows], y[rows], rng)
    return end_step


def new_run(init_seed, train_seed, sizes=(2, 32, 32, 2), dropout_p=0.1):
    model = MLP(list(sizes), np.random.default_rng(init_seed), dropout_p)
    return model, Adam(model.parameters()), np.random.default_rng(train_seed)


N = 100
model_a, optimiser_a, rng_a = new_run(1, 2)
run_training(model_a, optimiser_a, rng_a, 0, 2 * N)                     # run A: 2N steps in one go

model_b, optimiser_b, rng_b = new_run(1, 2)
run_training(model_b, optimiser_b, rng_b, 0, N)                         # run B: N steps ...
resume_path = os.path.join(checkpoint_dir, f"step_{N}.npz")
save_checkpoint(resume_path, model_b, optimiser_b, N, rng_b)
del model_b, optimiser_b, rng_b                                         # ... then everything is lost

model_c, optimiser_c, rng_c = new_run(999, 12345)                       # different seeds on purpose
resumed_step = load_checkpoint(resume_path, model_c, optimiser_c, rng_c)
run_training(model_c, optimiser_c, rng_c, resumed_step, resumed_step + N)

difference = max(float(np.abs(a.data - c.data).max()) for a, c in zip(model_a.parameters(), model_c.parameters()))
with np.load(resume_path) as saved:
    names = saved.files
print(f"checkpoint at step {resumed_step}: {os.path.getsize(resume_path):,} bytes holding {len(names)} arrays: {names[:4]} ...")
print(f"largest weight difference between the uninterrupted and the resumed run: {difference}")
assert all(np.array_equal(a.data, c.data) for a, c in zip(model_a.parameters(), model_c.parameters())), \
    "resumed training must reproduce uninterrupted training exactly"

**Expected behaviour.** The largest difference is exactly `0.0`. Comparing with `np.array_equal`, not with a tolerance, is legitimate here: both runs execute the same operations, in the same order, in the same process. Across different machines or library versions, small floating-point differences can appear, and a tolerance is then the right comparison.

## 8 · 🧪 Experiments

### 🧪 Experiment 1: what a tiny-batch failure looks like

The cell repeats the 16-example test with one deliberate bug. Run it with each option and compare the final loss with the healthy value, about 0.0004. Watch the last one carefully: the network memorises **shuffled** labels just as easily, so a passing tiny-batch test does not prove that the labels are right. Only validation data can show that.

In [ ]:
bug = "no zero_grad"  # 🧪 try "none", "detached logits", "learning rate 1e-5" or "shuffled labels"


def overfit_tiny_batch(bug, steps=300):
    inputs, targets = Z[tiny], y[tiny].copy()
    if bug == "shuffled labels":
        targets = np.random.default_rng(5).permutation(targets)
    model = MLP([2, 32, 32, 2], np.random.default_rng(3))
    optimiser = Adam(model.parameters(), lr=1e-5 if bug == "learning rate 1e-5" else 0.01)
    for _ in range(steps):
        model.train()
        if bug != "no zero_grad":
            optimiser.zero_grad()
        logits = model(Tensor(inputs))
        if bug == "detached logits":
            logits = Tensor(logits.data)
        loss = cross_entropy(logits, targets)
        loss.backward()
        optimiser.step()
    model.eval()
    accuracy = (model(Tensor(inputs)).data.argmax(axis=1) == y[tiny]).mean()
    return float(loss.data), accuracy


final_loss, true_label_accuracy = overfit_tiny_batch(bug)
print(f"bug = {bug!r}: final loss {final_loss:.4f}, accuracy on the true labels {true_label_accuracy:.0%}")

**What to notice.** Without `zero_grad`, every step adds the new gradient to all the old ones, so the updates soon stop matching the current batch and the loss stalls far above zero, at about 0.43. Detached logits leave the weights untouched, so the loss stays at its starting value of about 1.14. A learning rate of $10^{-5}$ barely moves the weights in 300 steps, and the loss is still about 1.01. With shuffled labels the loss still falls to almost zero, about 0.005, but the accuracy on the *true* labels is only about 38%: memorisation succeeded on the wrong targets.

### 🧪 Experiment 2: a learning rate that explodes

Plain SGD with a far too large learning rate makes the weights grow at every step. The gradient norm and the loss report it long before anything breaks, and the finite-loss guard in your `train_step` stops the run when the numbers finally overflow. Try the suggested values and compare the printed norms.

In [ ]:
learning_rate = 1e4  # 🧪 try 0.1, 10 and 100
boom_model = MLP([2, 32, 2], np.random.default_rng(5))
boom_optimiser, boom_rng = SGD(boom_model.parameters(), lr=learning_rate), np.random.default_rng(6)
with np.errstate(over="ignore", invalid="ignore"):          # overflow is the point of this experiment
    try:
        for step in range(1, 61):
            rows = train_rows[boom_rng.choice(train_rows.size, 32, replace=False)]
            loss_value = train_step(boom_model, boom_optimiser, Z[rows], y[rows], boom_rng)
            if step in (1, 2, 5, 10, 30, 60):
                print(f"step {step:2}: loss {loss_value:10.3g}, gradient norm {grad_norm(boom_model.parameters()):10.3g}")
        print(f"learning rate {learning_rate}: finished 60 steps")
    except FloatingPointError as error:
        print(f"learning rate {learning_rate}: stopped at step {step}: {error}")

**What to notice.** With a learning rate of 0.1 the loss settles near 0.45 and the gradient norm stays around 1. At 10 the loss jumps above 10 for a few steps before it recovers. At 100 the loss reaches about $10^{17}$ while staying finite, and then sinks to a useless plateau as most ReLU units die: a finite loss is *not* proof of a healthy run, which is why the gradient norm belongs in the log. At $10^4$ the weights overflow within about 50 steps, the loss becomes `nan`, and the guard stops training with a message instead of silently filling every weight with `nan`.

### 🧪 Experiment 3: a resumption that forgets something

Load the checkpoint from section 7 again, but leave one piece of state out, then train for $N$ more steps and compare with the uninterrupted run A.

In [ ]:
forget = "optimiser"  # 🧪 try "rng", "step" or None


def load_but_forget(path, model, optimiser, rng, forget):
    with np.load(path) as saved:
        for i, p in enumerate(model.parameters()):
            p.data[...] = saved[f"param_{i}"]
        if forget != "optimiser":
            optimiser.load_state_dict({key[4:]: saved[key] for key in saved.files if key.startswith("opt_")})
        if forget != "rng":
            rng.bit_generator.state = json.loads(str(saved["rng_state"]))
        return 0 if forget == "step" else int(saved["step"])


model_f, optimiser_f, rng_f = new_run(999, 12345)
start_step = load_but_forget(resume_path, model_f, optimiser_f, rng_f, forget)
run_training(model_f, optimiser_f, rng_f, start_step, start_step + N)
gap = max(float(np.abs(a.data - f.data).max()) for a, f in zip(model_a.parameters(), model_f.parameters()))
print(f"forget = {forget!r}: largest weight difference from the uninterrupted run = {gap:.3g}")

**What to notice.** Forgetting the optimiser state restarts Adam's moments at zero and its counter at 1, so the next steps are sized differently. Forgetting the random state changes every later batch and dropout mask. Forgetting the step restarts the learning-rate warm-up. Each omission produces a different model; only the complete checkpoint gives a difference of 0.

## 9 · Put it together: a monitored run that survives an interruption

The final run uses everything in this notebook. A 2 → 32 → 32 → 2 network with dropout 0.1 trains with Adam and a warm-up for 600 steps, which is 40 epochs of batches of 32. Every step logs the loss and the gradient norm, and every 50 steps the model is evaluated on the validation rows. Halfway through, the run is "interrupted": it saves a checkpoint, and a fresh process-like start loads it and finishes. An uninterrupted twin run checks that nothing changed.

In [ ]:
TOTAL_STEPS, INTERRUPT_AT = 600, 300


def monitored(model, optimiser, rng, start, end, log):
    for step in range(start + 1, end + 1):
        optimiser.lr = lr_at(step)
        rows = train_rows[rng.choice(train_rows.size, 32, replace=False)]
        log["loss"].append(train_step(model, optimiser, Z[rows], y[rows], rng))
        log["grad_norm"].append(grad_norm(model.parameters()))
        if step % 50 == 0:
            log["val"].append((step, *evaluate(model, val_rows)))


twin, twin_optimiser, twin_rng = new_run(20, 21)
initial_val_loss, initial_val_accuracy = evaluate(twin, val_rows)
initial_weights = [p.data.copy() for p in twin.parameters()]
twin_log = {"loss": [], "grad_norm": [], "val": []}
monitored(twin, twin_optimiser, twin_rng, 0, TOTAL_STEPS, twin_log)

first, first_optimiser, first_rng = new_run(20, 21)
log = {"loss": [], "grad_norm": [], "val": []}
monitored(first, first_optimiser, first_rng, 0, INTERRUPT_AT, log)
interrupt_path = os.path.join(checkpoint_dir, "interrupted.npz")
save_checkpoint(interrupt_path, first, first_optimiser, INTERRUPT_AT, first_rng)
del first, first_optimiser, first_rng

final, final_optimiser, final_rng = new_run(77, 78)
step_reached = load_checkpoint(interrupt_path, final, final_optimiser, final_rng)
monitored(final, final_optimiser, final_rng, step_reached, TOTAL_STEPS, log)

for step, val_loss, val_accuracy in log["val"][::2]:
    window = log["loss"][step - 50:step]
    print(f"step {step:3}: mean training loss {np.mean(window):.3f}, validation loss {val_loss:.3f}, "
          f"validation accuracy {val_accuracy:.1%}, gradient norm {log['grad_norm'][step - 1]:.2f}")

In [ ]:
steps = np.arange(1, TOTAL_STEPS + 1)
smoothed = np.convolve(log["loss"], np.ones(25) / 25, mode="valid")
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(steps[24:], smoothed, label="training loss (mean of 25 steps)")
ax.plot([s for s, _, _ in log["val"]], [v for _, v, _ in log["val"]], marker="o", label="validation loss")
ax.axvline(INTERRUPT_AT, linestyle="--", color="grey", label="save, reload, continue")
ax.set_xlabel("optimiser step")
ax.set_ylabel("cross-entropy")
ax.set_title("A monitored run with an interruption")
ax.legend()
plt.show()

In [ ]:
final_val_loss, final_val_accuracy = log["val"][-1][1:]
assert np.all(np.isfinite(log["loss"])) and np.all(np.isfinite(log["grad_norm"])), "every loss and gradient norm is finite"
assert all(np.array_equal(a.data, b.data) for a, b in zip(twin.parameters(), final.parameters())), \
    "the interrupted run must end with exactly the twin's weights"
assert log["loss"] == twin_log["loss"], "and it must have seen exactly the same losses"
assert any(not np.array_equal(p.data, w) for p, w in zip(final.parameters(), initial_weights)), "the weights changed"
assert np.mean(log["loss"][-50:]) < np.mean(log["loss"][:50]) - 0.1, "the training loss fell"
assert final_val_loss < initial_val_loss - 0.2, "the validation loss fell clearly"
assert final_val_accuracy >= 0.8, "validation accuracy should be well above the 51% majority baseline"
print(f"validation loss {initial_val_loss:.3f} -> {final_val_loss:.3f}; validation accuracy "
      f"{initial_val_accuracy:.1%} -> {final_val_accuracy:.1%}. All checks passed.")

**Expected behaviour.** The validation loss falls from about 0.92 before training to about 0.34, and validation accuracy rises from about 48% to about 89%, close to the ceiling that 10% of flipped labels allow. The gradient norm shrinks from about 1.1 to about 0.4 and never jumps. The interrupted run ends with exactly the twin's weights and saw exactly the same 600 losses, so the dashed line in the plot marks a restart that left no trace.

## Exercises

### Exercise 1: decoupled weight decay, `adamw_update`

Weight decay pulls every weight a little towards zero at each step. AdamW applies the decay directly to the parameter, separately from Adam's adaptive step: first `param = param - lr * weight_decay * param`, then the ordinary Adam update with the same `lr`. Write `adamw_update(param, grad, m, v, t, lr, weight_decay, beta1=0.9, beta2=0.999, eps=1e-8)`, returning `(new_param, new_m, new_v)`. You may call your `adam_update`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def adamw_update_reference(param, grad, m, v, t, lr, weight_decay, beta1=0.9, beta2=0.999, eps=1e-8):
    decayed = param - lr * weight_decay * param
    return adam_update_reference(decayed, grad, m, v, t, lr, beta1, beta2, eps)

In [ ]:
def adamw_update(param, grad, m, v, t, lr, weight_decay, beta1=0.9, beta2=0.999, eps=1e-8):
    """Decay the parameter towards zero, then take an ordinary Adam step."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
adamw_rng = np.random.default_rng(11)
adamw_update = practice("adamw_update", adamw_update, adamw_update_reference, cases=[
    (np.ones(3), np.array([0.5, -2.0, 1e-3]), np.zeros(3), np.zeros(3), 1, 0.1, 0.01),
    (adamw_rng.normal(size=(2, 2)), adamw_rng.normal(size=(2, 2)), np.zeros((2, 2)), np.zeros((2, 2)), 1, 0.01, 0.1),
    (np.ones(2), np.zeros(2), np.zeros(2), np.zeros(2), 1, 0.1, 0.5),       # edge case: decay alone moves the weights
])

### Exercise 2: `clip_by_global_norm`

Gradient clipping rescales all gradients together when their global norm exceeds a limit, keeping their direction. Write `clip_by_global_norm(grads, max_norm)` for a list of arrays: compute the global norm $n$; if $n > $ `max_norm`, return every array multiplied by `max_norm / n`; otherwise return copies of the arrays unchanged.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def clip_by_global_norm_reference(grads, max_norm):
    norm = float(np.sqrt(sum(float((g ** 2).sum()) for g in grads)))
    factor = max_norm / norm if norm > max_norm else 1.0
    return [g * factor for g in grads]

In [ ]:
def clip_by_global_norm(grads, max_norm):
    """Scale a list of gradient arrays so that their global L2 norm is at most max_norm."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
clip_by_global_norm = practice("clip_by_global_norm", clip_by_global_norm, clip_by_global_norm_reference, cases=[
    ([np.array([3.0, 0.0]), np.array([[0.0, 4.0]])], 1.0),     # norm 5, scaled down to 1
    ([np.array([0.3, 0.4])], 1.0),                             # norm 0.5: unchanged
    ([np.zeros(3)], 1.0),                                      # edge case: zero gradients
])

### Exercise 3: gradient accumulation, `accumulated_gradients`

Write `accumulated_gradients(model, X, y, micro_batch)`: clear the model's gradients once, then for each consecutive slice of `micro_batch` rows compute `cross_entropy(model(Tensor(X_slice)), y_slice)`, multiply it by `len(y_slice) / len(y)`, and call `backward()` without clearing in between. Return the list `[p.grad.copy() for p in model.parameters()]`. With that weighting, the accumulated gradient equals the gradient of the mean loss over the whole batch, even when the last slice is smaller; the probe checks this for a model without dropout.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def accumulated_gradients_reference(model, X, y, micro_batch):
    zero_grad(model.parameters())
    for start in range(0, len(y), micro_batch):
        X_part, y_part = X[start:start + micro_batch], y[start:start + micro_batch]
        (cross_entropy(model(Tensor(X_part)), y_part) * (len(y_part) / len(y))).backward()
    return [p.grad.copy() for p in model.parameters()]

In [ ]:
def accumulated_gradients(model, X, y, micro_batch):
    """Gradients of the mean loss over (X, y), accumulated over micro-batches of micro_batch rows."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
def accumulation_probe(fn, seed, n_rows, micro_batch):
    """Probe: accumulated gradients on a fresh model without dropout."""
    model = MLP([2, 8, 2], np.random.default_rng(seed))
    model.eval()
    rows = train_rows[:n_rows]
    return fn(model, Z[rows], y[rows], micro_batch)


accumulated_gradients = practice_via("accumulated_gradients", accumulated_gradients, accumulated_gradients_reference,
                                     accumulation_probe, cases=[(0, 32, 8), (1, 30, 8), (2, 5, 16)])

check_model = MLP([2, 8, 2], np.random.default_rng(0))
check_model.eval()
zero_grad(check_model.parameters())
cross_entropy(check_model(Tensor(Z[train_rows[:30]])), y[train_rows[:30]]).backward()
whole_batch = [p.grad.copy() for p in check_model.parameters()]
pieces = accumulated_gradients(check_model, Z[train_rows[:30]], y[train_rows[:30]], 8)
print("accumulated over micro-batches of 8 (8 + 8 + 8 + 6 rows) equals the whole batch of 30:",
      all(np.allclose(a, b) for a, b in zip(whole_batch, pieces)))

## Common errors and debugging

- **The loss stalls and the updates look erratic.** `zero_grad()` is missing, so gradients from earlier batches pile up.
- **Validation scores change from one call to the next.** The model is still in training mode, so dropout is active; call `model.eval()` before evaluating, and `model.train()` before the next update.
- **`model.eval()` was called, yet memory keeps growing during validation.** Evaluation mode does not stop gradient recording. In frameworks such as PyTorch, wrap validation in `torch.no_grad()`.
- **`FloatingPointError: non-finite training loss`.** The learning rate is too large, the inputs contain `nan` or `inf`, or the data were not standardised. Lower the learning rate and inspect the batch.
- **A resumed run drifts away from the original.** The checkpoint lacks the optimiser state, the step, or the random state. Save all of them, and restore the random state *after* building the fresh model, because building it draws random numbers.
- **The optimiser updates tensors that the model no longer uses.** Parameters were replaced with new objects (`p.data = ...` on a copy, or a rebuilt layer) after the optimiser was created. Load weights in place with `p.data[...] = ...`.
- **`ValueError: Object arrays cannot be loaded when allow_pickle=False`.** Something saved a Python object instead of numbers or text. Store state as arrays or JSON text; do not turn pickling on for files you did not write.
- **A run that makes no updates.** Fewer examples than one micro-batch under a drop-last policy; check the update arithmetic before long runs.

## Summary

- Modules own their parameters and list them with `parameters()`; a mode flag switches layers such as dropout between training and evaluation.
- One update is: training mode, clear gradients, forward pass and loss, finite check, backward, step. Validation is: evaluation mode, forward pass, nothing else.
- Adam scales each parameter's step by running averages of the gradient and its square; bias correction fixes their zero start, so $t$ is part of the state.
- Diagnose on a tiny batch first, then log the loss, the gradient norm and the weight change; a finite loss is not enough.
- Budget in updates and examples, and count memory beyond the weights: training state alone is 16 to 32 bytes per parameter.
- A checkpoint that holds the weights, the optimiser state, the step and the random state resumes training exactly.

## Next

Notebook 4 trains with validation-based checkpoint selection, evaluates the chosen model once on the test rows, saves it as a self-contained artefact, proves that the reloaded model reproduces its predictions, and sets up your capstone experiment.